# PixAI Core · Colab آزمایشی

این Notebook برای اجرای آزمایشی یک مدل گفت‌وگویی در محیط ابری Google Colab است. مدل روی Runtime ابری دریافت و اجرا می‌شود؛ چیزی روی گوشی نصب نمی‌شود.

**نکته:** این نسخه آزمایشی و موقت است. با قطع شدن Runtime، اجرای مدل متوقف می‌شود و ممکن است لازم باشد مدل دوباره بارگذاری شود. این Notebook یک API دائمی یا سرویس ۲۴ساعته ایجاد نمی‌کند.

مدل پیش‌فرض: `Qwen/Qwen3-1.7B`

## ۱) فعال کردن GPU

در Colab از منوی **Runtime → Change runtime type**، گزینه GPU را انتخاب کن (اگر در حسابت در دسترس بود).

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU در دسترس نیست. برای این مدل، اجرای CPU ممکن است بسیار کند باشد.')

## ۲) نصب کتابخانه‌ها

این مرحله کتابخانه‌های لازم را در Runtime ابری نصب می‌کند.

In [ ]:
%pip -q install -U transformers accelerate

## ۳) بارگذاری مدل

وزن‌های مدل در فضای ابری Colab دریافت می‌شوند. حجم دانلود و زمان بارگذاری به سرعت اینترنت و منابع Runtime بستگی دارد.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = 'Qwen/Qwen3-1.7B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

load_kwargs = {'device_map': 'auto'}
if torch.cuda.is_available():
    load_kwargs['torch_dtype'] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
print('PixAI Core آماده آزمایش است.')

## ۴) گفت‌وگو با PixAI Core

سلول زیر یک گفت‌وگوی ساده داخل خود Notebook اجرا می‌کند. برای پایان دادن بنویس `خروج` و برای پاک کردن سابقه بنویس `پاک‌کردن`.

In [ ]:
SYSTEM_PROMPT = 'تو PixAI Core هستی؛ یک دستیار فارسی‌زبان، دقیق، دوستانه و مسئولیت‌پذیر. اگر چیزی را نمی‌دانی، صادقانه بگو.'
messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]

def ask_pixai(user_text, max_new_tokens=384):
    global messages
    messages.append({'role': 'user', 'content': user_text})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = output[0][inputs['input_ids'].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    messages.append({'role': 'assistant', 'content': answer})
    return answer

print('PixAI Core فعال شد. برای خروج، «خروج» را بنویس.')
while True:
    user_text = input('شما: ').strip()
    if user_text == 'خروج':
        print('گفت‌وگو پایان یافت.')
        break
    if user_text == 'پاک‌کردن':
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
        print('سابقه گفت‌وگو پاک شد.')
        continue
    if not user_text:
        continue
    try:
        print('PixAI:', ask_pixai(user_text))
    except Exception as error:
        print('خطا:', repr(error))
        break

## محدودیت‌های این نسخه

- این فقط محیط آزمایش چت است؛ هنوز به اپلیکیشن PixAI یا ImageEngine متصل نشده.
- Colab رایگان GPU و زمان اجرای پیوسته را تضمین نمی‌کند.
- با پایان Runtime، مدل از حافظه خارج می‌شود و برای اجرای دوباره باید بارگذاری شود.
- برای آموزش اختصاصی با دیتاست پروژه، مرحله جداگانه‌ای لازم است؛ این Notebook فعلاً مدل پایه را اجرا می‌کند.